# Claude Web Search Tool

- The Claude web search tool is provided by Claude and runs on Claude's servers, so we only send the schema and read the result.
- The tool must be enabled by an admin in the org console before it can be used.
- Key response blocks to understand:
  - text blocks: Claude's own explanation of what it is doing,
  - ServerToolUseBlock: the exact search query Claude issued,
  - WebSearchToolResultBlock: the returned search results payload,
  - WebSearchResultBlock: each individual result with title and URL,
  - citation blocks: supporting text that Claude uses to back its statements.
- This notebook shows the basic schema usage and how Claude can call a web search tool to answer user questions.


In [ ]:
# Load env variables and create client
from dotenv import load_dotenv
from anthropic import Anthropic

load_dotenv()

client = Anthropic()
model = "claude-sonnet-4-5"

In [ ]:
# Helper functions
from anthropic.types import Message


def add_user_message(messages, message):
    user_message = {
        "role": "user",
        "content": message.content if isinstance(message, Message) else message,
    }
    messages.append(user_message)


def add_assistant_message(messages, message):
    assistant_message = {
        "role": "assistant",
        "content": message.content if isinstance(message, Message) else message,
    }
    messages.append(assistant_message)


def chat(messages, system=None, temperature=1.0, stop_sequences=[], tools=None):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature,
        "stop_sequences": stop_sequences,
    }

    if tools:
        params["tools"] = tools

    if system:
        params["system"] = system

    message = client.messages.create(**params)
    return message


def text_from_message(message):
    return "\n".join([block.text for block in message.content if block.type == "text"])

In [ ]:
web_search_schema = {
    "type": "web_search_20250305",
    "name": "web_search",
    "max_uses": 5,
    "allowed_domains": ["nih.gov"],
}

In [ ]:
messages = []
add_user_message(
    messages,
    """
    What's the best exercise for gaining leg muscle?
    """,
)
response = chat(messages, tools=[web_search_schema])
response